In [1]:
%config Completer.use_jedi = False

In [2]:
# load module
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import copy
import warnings
import torch
import optuna
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger


from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
#from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

import tensorflow as tf 
import tensorboard as tb 
tf.io.gfile = tb.compat.tensorflow_stub.io.gfile

import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")  # avoid printing out absolute paths
plt.rcParams['font.family'] = 'NanumGothic'
#plt.rcParams['font.sans-serif'] = ['NanumGothic.ttf', 'sans-serif']


/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.36ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.36ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(


In [3]:
# functions
def ewma_data_processing(path = '../../sample_table/ewma_6h_scaling.csv'):
    data = pd.read_csv(path)
    data.drop('Unnamed: 0.2'   , axis=1 , inplace=True)
    data.drop('Unnamed: 0.1'   , axis=1 , inplace=True)
    data.drop('Unnamed: 0'   , axis=1 , inplace=True)
    data['REG_DTIME'] = pd.to_datetime(data['REG_DTIME'])
    data['DOW'] = data['REG_DTIME'].dt.dayofweek
    data['HOD'] = data['REG_DTIME'].dt.hour
    data["time_idx"] =  \
    (data["REG_DTIME"].dt.month) * data["REG_DTIME"].dt.daysinmonth * 24  + \
    data["REG_DTIME"].dt.day * 24  + \
    data["REG_DTIME"].dt.hour 
    data["time_idx"] -= data["time_idx"].min()
    data['h_dong'] = data['h_dong'].astype(str)
    data['DOW'] = data['DOW'].astype(str)
    data['HOD'] = data['HOD'].astype(str)
    data['precip_form'] = data['precip_form'].astype(str)
    data['isHoliday'] = data['isHoliday'].astype(str)

    return data

def org_data_processing(path = '../sample_table/total_data.csv'):
    data = pd.read_csv(path)
    data.drop('Unnamed: 0'   , axis=1 , inplace=True)
    data['REG_DTIME'] = pd.to_datetime(data['REG_DTIME'])
    data['DOW'] = data['REG_DTIME'].dt.dayofweek
    data['HOD'] = data['REG_DTIME'].dt.hour
    data["time_idx"] =  \
    (data["REG_DTIME"].dt.month) * data["REG_DTIME"].dt.daysinmonth * 24  + \
    data["REG_DTIME"].dt.day * 24  + \
    data["REG_DTIME"].dt.hour 
    data["time_idx"] -= data["time_idx"].min()
    data['h_dong'] = data['h_dong'].astype(str)
    data['DOW'] = data['DOW'].astype(str)
    data['HOD'] = data['HOD'].astype(str)
    data['precip_form'] = data['precip_form'].astype(str)
    data['isHoliday'] = data['isHoliday'].astype(str)

    return data

def get_training(data , max_prediction_length, max_encoder_length):
    # traing data 생성
    max_prediction_length = max_prediction_length
    max_encoder_length = max_encoder_length
    training_cutoff = data["time_idx"].max() - max_prediction_length

    training = TimeSeriesDataSet(
        data[lambda x : x.time_idx <= training_cutoff],
        time_idx = "time_idx",
        target = "count",
        group_ids = ['h_dong'],
        min_encoder_length=max_encoder_length // 2,  # keep encoder length long (as it is in the validation set)
        max_encoder_length=max_encoder_length,
        min_prediction_length=1,
        max_prediction_length=24,
        static_categoricals=["h_dong"],
        time_varying_known_categoricals=["HOD", "DOW" , 'isHoliday'],
        time_varying_known_reals=['pops'],
        #variable_groups={"special_days": special_days},  # group of categorical variables can be treated as one variable
        time_varying_unknown_categoricals=['precip_form'],
        time_varying_unknown_reals=['count','windspd' , 'temp' ,'precip'],
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        #allow_missing=True,
        allow_missing_timesteps = True)
    return training

def truncate(num, rat):
    if (num - int(num)) < rat:
        return int(num)
    else:
        return int(num)+1
    
# boolean으로 계산시에는 call하는 함수에서 categoize
def perf_measure_boolean(y_actual, y_pred):
    # y_actual value
    # y_pred  array
    # numberical measuer
    TP =  FP = TN = FN = 0
    
    if y_actual == 1 and sum(y_pred) > 0:
        TP += 1
    if y_actual == 1 and sum(y_pred) == 0:
        FN += 1
    if y_actual == 0 and sum(y_pred) > 0:
        FP += 1
    if y_actual == 0 and sum(y_pred) == 0:
        TN += 1

    return TP, FN ,FP, TN

def perf_measure_numerial(y_actual, y_pred):
    # y_actual one array
    # y_pred  array
    # numberical measuer : set class로 and 연산
    TP =  FP = TN = FN = 0
    inter = np.intersect1d(y_actual , y_pred)

    if len(y_pred) == 1:
        if y_actual[0] == y_pred[0] and y_actual[0] > 0:
            TP += 1
        if y_actual[0] > 0 and y_actual[0] != y_pred[0]:
            FN += 1
        if y_actual[0] == 0 and y_pred[0] > 0 :
            FP += 1
        if y_actual[0] == y_pred[0] == 0:
            TN += 1

    else:
        if len(inter) == 0:
            if y_actual[0] > 0:
                FN +=1
            if y_actual[0] == 0 and sum(y_pred) > 0:
                FP += 1

                
        if len(inter) == 1:
            if y_actual[0] > 0:
                TP +=1
            if y_actual[0] == 0 and sum(y_pred) > 0:
                FP += 1
            elif y_actual[0] == 0:
                TN += 1
    return TP, FN ,FP, TN 

def boolean(num):
    if num == 0:
        return 0
    else:
        return 1

In [4]:
ewma_data = ewma_data_processing('../../training_data/ewma_6h_scaling_round.csv')
ewma_training = get_training(ewma_data,24*30, 24*30*2)



In [5]:
batch_size= 32
ewma_dataloader = ewma_training.to_dataloader(batch_size)
ewma_validation = TimeSeriesDataSet.from_dataset(ewma_training, ewma_data, predict=True, stop_randomization=True)
ewma_train_dataloader = ewma_training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
ewma_val_dataloader = ewma_validation.to_dataloader(train=False, batch_size=batch_size*100, num_workers=0)

In [6]:
epoch = 300
hidden_size = 16
atten_head = 4

pl.seed_everything(42)

early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min")
lr_logger = LearningRateMonitor()  
logger = TensorBoardLogger('test')
trainer = pl.Trainer(
    max_epochs=epoch,
    gpus=0,
    gradient_clip_val=0.1,
    limit_train_batches=5,  
    callbacks=[lr_logger],
    logger=logger,
)

tft = TemporalFusionTransformer.from_dataset(
    ewma_training,
    learning_rate=0.03,
    hidden_size=hidden_size,  # most important hyperparameter apart from learning rate
    # number of attention heads. Set to up to 4 for large datasets
    attention_head_size=atten_head,
    dropout=0.1,  # between 0.1 and 0.3 are good values
    hidden_continuous_size=hidden_size,  # set to <= hidden_size
    output_size=7,  # 7 quantiles by default
    loss=QuantileLoss(),
    # reduce learning rate if no improvement in validation loss after x epochs
    reduce_on_plateau_patience=4,
)

tft.hparams['optimizer'] = 'RAdam'

Global seed set to 42
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [7]:
trainer.fit(
    tft,
    train_dataloaders=ewma_train_dataloader,
    val_dataloaders=ewma_val_dataloader
)
trainer.checkpoint_callback.best_model_path


   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 466   
3  | prescalers                         | ModuleDict                      | 288   
4  | static_variable_selection          | VariableSelectionNetwork        | 3.8 K 
5  | encoder_variable_selection         | VariableSelectionNetwork        | 8.8 K 
6  | decoder_variable_selection         | VariableSelectionNetwork        | 2.9 K 
7  | static_context_variable_selection  | GatedResidualNetwork            | 1.1 K 
8  | static_context_initial_hidden_lstm | GatedResidualNetwork            | 1.1 K 
9  | static_context_initial_cell_lstm   | GatedResidualNetwork            | 1.1 

Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=300` reached.


'test/lightning_logs/version_8/checkpoints/epoch=299-step=1500.ckpt'